In [ ]:
from src import available_models, get_model, MODEL_REGISTRY
from src.discrete.quantizer import MultiScaleResidualQuantizer, SimVQ, QINCoVectorQuantizer2, VectorQuantizer2, ResidualQuantizer, GroupedVQ, LookupFreeQuantizer, FiniteScalarQuantizer


import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST, CIFAR100
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
from dataset import MergedMedMNIST

### Prep Dataset ###
tensor_transforms = transforms.Compose(
    [
        transforms.Resize((256,256)),
        transforms.ToTensor(),
    ]
)

# train_set = MergedMedMNIST(root="/vol/miltank/users/bubeckn/MedMNIST", split="train", transform=tensor_transforms, download=True, size=224)
# eval_set = MergedMedMNIST(root="/vol/miltank/users/bubeckn/MedMNIST", split="val", transform=tensor_transforms, download=True, size=224)
# test_set = MergedMedMNIST(root="/vol/miltank/users/bubeckn/MedMNIST", split="test", transform=tensor_transforms, download=True, size=224)


train_set = CIFAR100(root=".", train=True, transform=tensor_transforms, download=True)
test_set = CIFAR100(root=".", train=False, transform=tensor_transforms, download=True)

### Set Device ###
device = "cuda" if torch.cuda.is_available() else "cpu"

attention mode is flash


ImportError: cannot import name 'Encoder' from 'src.modules' (unknown location)

In [ ]:
def train(model,
          train_set,
          test_set,
          batch_size,
          num_epochs,
          evaluation_iterations):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)

    trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=8)

    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    train_losses = []
    evaluation_losses = []

    steps_per_epoch = len(trainloader)
    total_steps = steps_per_epoch * num_epochs
    step_counter = 0
    train_loss_running = []
    evaluation_loss_running = []

    with tqdm(total=num_epochs, desc="Epochs") as epoch_bar:
        for epoch in range(num_epochs):
            model.train()
            with tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{num_epochs} | loss: N/A", leave=False) as pbar:
                for images, labels in trainloader:
                    images = images.to(device)

                    recon, _ = model(images)
                    loss = F.mse_loss(recon, images) + 0# qloss
                    # Update tqdm description with current loss value
                    pbar.set_description(f"Epoch {epoch+1}/{num_epochs} | loss: {loss.item():.4f}")

                    train_loss_running.append(loss.item())

                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()

                    step_counter += 1
                    pbar.update(1)

                    # Evaluate every evaluation_iterations global step
                    if step_counter % evaluation_iterations == 0:
                        model.eval()
                        evaluation_loss_running.clear()
                        example_shown = False
                        with torch.no_grad():
                            for images_eval, labels_eval in testloader:
                                images_eval = images_eval.to(device)
                                recon_eval, _ = model(images_eval)
                                loss_eval = F.mse_loss(recon_eval, images_eval) + 0# qloss_eval
                                evaluation_loss_running.append(loss_eval.item())
                                
                                if not example_shown:

                                    def prep(img):
                                        img = img.detach().cpu()
                                        return img.squeeze(0) if img.shape[0] == 1 else img.permute(1, 2, 0)

                                    inp = prep(images_eval[0])
                                    rec = prep(recon_eval[0])
                                    cmap = "gray" if inp.ndim == 2 else None

                                    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
                                    titles = ["Input", "Reconstruction"]

                                    for ax, data, title in zip(axes, [inp, rec], titles):
                                        ax.imshow(data, cmap=cmap)
                                        ax.set_title(title)
                                        ax.axis("off")

                                    plt.suptitle(f"Eval Example at Step {step_counter}")
                                    plt.tight_layout()
                                    plt.show()

                                    example_shown = True

                        train_loss_val = np.mean(train_loss_running) if train_loss_running else 0.0
                        evaluation_loss_val = np.mean(evaluation_loss_running) if evaluation_loss_running else 0.0
                        train_losses.append(train_loss_val)
                        evaluation_losses.append(evaluation_loss_val)
                        train_loss_running = []
                        model.train()
            epoch_bar.update(1)

    print("Final Training Loss", train_losses[-1] if train_losses else "No log")
    print("Final Evaluation Loss", evaluation_losses[-1] if evaluation_losses else "No log")

    return model, train_losses, evaluation_losses

In [ ]:
from src.continuous.vae_models import AutoencoderKL_f16


# encoder = Encoder(img_size=64, dims=2, double_z=False, ch_mult=(1, 2, 4))
# decoder = Decoder(img_size=64, dims=2, double_z=False, ch_mult=(1, 2, 4))
# quantizer = SimVQ(in_channels=16, n_e=8192, e_dim=3)
# quantizer = GroupedVQ(
#     quantizer_class=ResidualQuantizer,
#     in_channels=4,
#     quantizer_kwargs_list=[
#         {"quantizer_class": VectorQuantizer2, "num_quantizers": 3, "quantizer_kwargs_list": [{"n_e": 8192, "e_dim": 4}, {"n_e": 4096, "e_dim": 4}, {"n_e": 2048, "e_dim": 4}]}
#     ] * 4,
#     groups=4,
# )

# quantizer = LookupFreeQuantizer(
#     token_bits=10,
# )

# quantizer = FiniteScalarQuantizer(
#     levels=[15]*64,
# )

# quantizer = MultiScaleResidualQuantizer(
#     n_e=8192,
#     e_dim=16,
#     v_patch_nums=(1, 2, 3, 4, 5, 6, 8, 12, 16),
#     quant_resi=0.5,
#     share_quant_resi=4,
#     using_znorm=False,
# )

# vq_model = VQModel(encoder=encoder, decoder=decoder, quantizer=quantizer)
vq_model = get_model("token.maetok.b_128")
# print(f"VQ model codebook size: {vq_model.quantizer.n_e}, embed dim: {vq_model.embed_dim}, z channels: {vq_model.z_channels}")

linear_ae, train_losses, evaluation_losses = train(vq_model,
                                                   train_set=train_set,
                                                   test_set=test_set,
                                                   batch_size=8,
                                                   num_epochs=10,
                                                   evaluation_iterations=1000)

In [ ]:
generated_index = 150
image, label = test_set[generated_index]

# Add batch dimension, move to device
image = image.unsqueeze(0).to(device)

# Forward pass
recon, _ = vq_model(image)

# Move to CPU, remove batch dim → shape: (C, H, W)
recon = recon.cpu().detach().squeeze()

# Convert to H×W×C (RGB) or H×W (grayscale)
if recon.ndim == 3:                 # C,H,W
    if recon.shape[0] == 1:         # grayscale
        recon = recon.squeeze(0)    # → H,W
        cmap = "gray"
    else:                           # RGB
        recon = recon.permute(1, 2, 0).numpy()
        cmap = None
else:
    cmap = "gray"

plt.imshow(recon, cmap=cmap)
plt.title("RVQVAE")
plt.axis("off")
plt.show()